In [18]:
%load_ext autoreload
%autoreload 2

from Utils import Notebook
from Utils import Tex
from Utils import Graphs
from IPython.display import Math, display
import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg as la
import json

from IPython.display import display, Math, Latex,Markdown

import ControllerDesigner, Engine


Notebook.setup()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


LaTeX has been enabled for text rendering.


### Definição da Planta

In [19]:
experiment_config_filename = "sys-01"
config_path = fr"../../data/lit-systems/{experiment_config_filename}.json"

with open(config_path, "r", encoding="utf-8") as f:
  plant_data = json.load(f)

plant_matrices = plant_data["system_matrices"]
A = np.array(plant_matrices["A"], dtype=np.float64)
B = np.array(plant_matrices["B"], dtype=np.float64)
C = np.array(plant_matrices["C"], dtype=np.float64)

results = ControllerDesigner.check_system_properties(A, B, C)

print("=" * 60)
print("SYSTEM PROPERTIES")
print("=" * 60)
print(
    f"Controllability rank : {results['controllability_rank']} / {A.shape[0]}")
print(f"Observability rank   : {results['observability_rank']} / {A.shape[0]}")
print(f"Controllable         : {results['controllable']}")
print(f"Observable           : {results['observable']}")
print(f"Stabilizable         : {results['stabilizable']}")
print(f"Detectable           : {results['detectable']}")

print("\nEigenvalues of A:")
for eig in results["eigenvalues"]:
  print(f"  {eig:.2f}")
print("=" * 60)

print("\nEquações que regem a dinâmica da planta:")
A_latex_expr = Tex.mat2tex(A)
B_latex_expr = Tex.mat2tex(B)
C_latex_expr = Tex.mat2tex(C)
display(Math(rf"\dot{{x}}(t) = {A_latex_expr} x(t) + {B_latex_expr} u(t)"))
display(Math(rf"y(t) = {C_latex_expr} x(t)"))

sim_engine = Engine.Engine(dll_path="dll/petc_for_lit_systems.dll")
sim_engine.load_lit_model(config_path)

print(f"\n[Engine] Modelo carregado com sucesso via DLL: {config_path}")
print(
    f"Dimensões detectadas -> Estados (nx): {sim_engine.nx}, Entradas (nu):"
    f" {sim_engine.nu}, Saídas (ny): {sim_engine.ny}"
)

SYSTEM PROPERTIES
Controllability rank : 2 / 2
Observability rank   : 2 / 2
Controllable         : True
Observable           : True
Stabilizable         : True
Detectable           : True

Eigenvalues of A:
  0.20+1.99j
  0.20-1.99j

Equações que regem a dinâmica da planta:


<IPython.core.display.Math object>

<IPython.core.display.Math object>


[Engine] Modelo carregado com sucesso via DLL: ../../data/lit-systems/sys-01.json
Dimensões detectadas -> Estados (nx): 2, Entradas (nu): 1, Saídas (ny): 1


### Co-projeto do Controlador baseado em Eventos

In [20]:
h = 1e-1
lambd = 1e-3

upsilon1 = 1e-3
upsilon2 = 1e-3
upsilon3 = 1e-2

ctrl_params = {'A': A, 'B': B, 'C': C,  'h': h, 'λ': lambd,
               'υ1': upsilon1, 'υ2': upsilon2, 'υ3': upsilon3}

synth_res = ControllerDesigner.synthesize_controller_setm(
    ctrl_params, eps=1e-6, verbose=False)

if synth_res is None:
  print("ERRO: Síntese Infeasible ou falha na recuperação das matrizes.")
else:
  K = synth_res['controller']['K']
  P = synth_res['functional']['P']
  R = synth_res['functional']['R']
  Ψ = synth_res['etm']['Ψ']
  Ξ = synth_res['etm']['Ξ']
  eig_cl = np.linalg.eigvals(A + B @ K)
  print("=== SÍNTESE CONCLUÍDA COM SUCESSO ===")
  print(f"Status do Solver: {synth_res['solver_status']}")
  print(f"Autovalores Nominais de Malha Fechada: {np.round(eig_cl, 4)}")

  display(Math(rf"""
    \begin{{aligned}}
        K &= {Tex.mat2tex(K)}, \qquad 
        P = {Tex.mat2tex(P)}, \qquad 
        R = {Tex.mat2tex(R)} \\[10pt]
        \Psi &= {Tex.mat2tex(Ψ)}, \qquad 
        \Xi = {Tex.mat2tex(Ξ)}
    \end{{aligned}}
    """))

=== SÍNTESE CONCLUÍDA COM SUCESSO ===
Status do Solver: optimal
Autovalores Nominais de Malha Fechada: [-0.9761+1.4863j -0.9761-1.4863j]


<IPython.core.display.Math object>

### Projeto do Observador Impulsivo

In [39]:
observer_params = {"A": A, "C": C, "lambda_obs": 5.0}
obs_result = ControllerDesigner.synthesize_observer_luenberger_min_peaking(
    observer_params)
if obs_result is not None:
  print("=== SÍNTESE DO OBSERVADOR COM SUCESSO ===")
  L = obs_result['observer']['L']
  display(Math(rf"""L = {Tex.mat2tex(L)}"""))
  print("Fator de pico:",
        obs_result['performance']['theoretical_peaking_factor'])
else:
  print("ERRO: Síntese Infactível.")

6.0000000040695065
712.0968423224471
=== SÍNTESE DO OBSERVADOR COM SUCESSO ===


<IPython.core.display.Math object>

Fator de pico: 10.894163858392012


In [ ]:
def format_matrix_to_cpp(mat, name="X", scientific=True):
  """Converte um vetor ou matriz numpy para o formato C++ com chave única:

  X = {val1, val2, val3, ...};
  """
  mat = np.atleast_2d(mat)
  flat_vals = mat.flatten()

  fmt = "{:.2e}" if scientific else "{:.4f}"
  vals_str = ", ".join([fmt.format(val) for val in flat_vals])

  return f"{name} = {{{vals_str}}};"


# Extração das variáveis do novo escopo Dual-Channel
Ψ = synth_res["etm"]["Ψ"]
Ξ = synth_res["etm"]["Ξ"]
K = synth_res["controller"]["K"]
L = obs_result["observer"]["L"]

# Exibição no console / Jupyter no formato exato solicitado
print(format_matrix_to_cpp(Ξ, name="Ξ"))
print(format_matrix_to_cpp(Ψ, name="Ψ"))
print(format_matrix_to_cpp(K, name="K"))
print(format_matrix_to_cpp(L, name="L"))

Ξ = {2.93e+03, -7.19e+03, -7.19e+03, 2.04e+04};
Ψ = {5.65e+03, 6.83e+02, 6.83e+02, 6.72e+03};
K = {8.38e-01, -2.35e+00};
L = {3.01e+02, 1.70e+03};
